In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from llm_utils import *
import pandas as pd

/home/isapc/Tesis/unsloth_env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Alert: Check if the ##Response is being generated correctly. If not, check the model name and the tokenizer.


In [3]:
# model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
# model_name = "unsloth/DeepSeek-R1-Distill-Qwen-1.5B-unsloth-bnb-4bit"
# model_name = "unsloth/phi-4-unsloth-bnb-4bit"
# model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
# model_name = "unsloth/Qwen3-8B-unsloth-bnb-4bit"
# model_name= "unsloth/Qwen3-14B-unsloth-bnb-4bit"
model_name = "unsloth/DeepSeek-R1-Distill-Llama-8B-unsloth-bnb-4bit"

In [4]:
llm_model = LLMModel(model_name=model_name)

# llm_model.load_pretrained_model(max_seq_length=1024 * 5)
# Voy a hacer test usando kfold cross validation, así que no necesito cargar el dataset completo, sino solo el fold de test
llm_model.load_test_csv_dataset("/home/isapc/Tesis/CohesentiaExperiments/preprocessing/create_binary_reasons_dataset/cohesentia_train_cot_balanced.csv")

   [ID]                                               text  [score]  \
0     2  Title: Deadly Text: Once upon a time, there we...        0   
1     5  Title: What is your favorite place, city or co...        1   
2     7  Title: A Life-Changing Journey Text: It was a ...        1   
3     9  Title: Helping a friend Text: One summers morn...        0   
4    10  Title: The Desperate Quest Text: When the prin...        1   

                                               [COT]  [num_annotators]  \
0                               Error: Invalid Score                 5   
1  1. Overall Coherence:\n\t- The story is not co...                 5   
2  1. Overall Coherence:\n\t- The story is not co...                 5   
3                               Error: Invalid Score                 5   
4  1. Overall Coherence:\n\t- The story is not co...                 5   

                     [holistic_scores_per_annotator]  [R1]  [R2]  [R3]  [R4]  \
0                         {'annot0': 1, 'annot1'

In [5]:
llm_model.perform_stratified_cv(llm_model.test_dataset)

Se crearon exitosamente 10 folds.


In [6]:
llm_model.load_lora_model(load_path='./DeepSeek-R1-Distill-Llama-8B-acc_89_holistic_consensus_score', max_seq_length=1024 * 5)

==((====))==  Unsloth 2026.6.7: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 4060 Ti. Num GPUs = 1. Max memory: 15.996 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 291/291 [00:02<00:00, 144.94it/s]
Unsloth: Will load ./DeepSeek-R1-Distill-Llama-8B-acc_89_holistic_consensus_score as a legacy tokenizer.
Unsloth 2026.6.7 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


LoRA model loaded from ./DeepSeek-R1-Distill-Llama-8B-acc_89_holistic_consensus_score


In [7]:
generation_prompt_ds = """
<｜begin▁of▁sentence｜><｜User｜>
You are an AI system tasked with evaluating the textual coherence of a short story based on its title and content. Your job is to perform a sentence-by-sentence analysis of the story and identify any coherence issues. Each problematic sentence must be tagged with the appropriate reason code, explained below.

## Instruction:
Analyze the given story sentence by sentence. Identify any coherence issues and assign the relevant tags from the list below. Each problematic sentence should include a short explanation and be marked with one or more of the following reason tags:

### Reason Tags:
<r1>1</r1> - the sentence doesn't make sense
<r2>1</r2> - the sentence discusses an entity which has not been introduced yet
<r3>1</r3> - the relation between this sentence and previous ones doesn't make sense
<r4>1</r4> - the sentence contains information inconsistent with previous presented data
<r5>1</r5> - the sentence contains information inconsistent with the knowledge about the world
<r6>1</r6> - the sentence is not relevant to the title
<r7>1</r7> - the sentence is not relevant to previous data in the story


## Scoring criteria:
- <score>1</score> – The story is coherent overall.
- <score>0</score> – The story contains significant coherence issues that impact understanding or narrative flow.

## Response Format:
1. Overall Coherence:
   - [Brief assessment of whether the story is coherent or not]
   
2. Problematic sentences:
   - "[Exact problematic sentence]" !Important: don not repeat the sentences, each sentence must be unique!
     - Reason: [Brief explanation]
     - Tags: <rX>1</rX>
     
3. Conclusion:
   - [Summary conclusion about the story’s coherence]
   
4. Score:
   - <score>[0 or 1]</score>

## Input:
{}
<｜Assistant｜>

## Response:
"""

In [8]:
import sys

class LoggerTee:
    def __init__(self, filename):
        self.terminal = sys.stdout
        self.log = open(filename, "a", encoding="utf-8")

    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)
        self.log.flush()  # Fuerza la escritura inmediata en el disco duro

    def flush(self):
        self.terminal.flush()
        self.log.flush()

class ErrorTee:
    def __init__(self, filename):
        self.stderr = sys.stderr
        self.log = open(filename, "a", encoding="utf-8")

    def write(self, message):
        self.stderr.write(message)
        self.log.write(message)
        self.log.flush()

    def flush(self):
        self.stderr.flush()
        self.log.flush()

In [ ]:
# 1. Definir el nombre del archivo de log (puedes meterle un timestamp)
log_filename = f"eval_coherence_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
print(f"Guardando todos los logs de ejecución en: {log_filename}")

# 2. Respaldar salidas originales
original_stdout = sys.stdout
original_stderr = sys.stderr

try:
    # 3. Activar el Logger (A partir de aquí, todo se duplica al archivo)
    sys.stdout = LoggerTee(log_filename)
    sys.stderr = ErrorTee(log_filename)

    # 4. Lanzar tu proceso completo
    all_folds_results = llm_model.run_cross_validation_evaluation(
        prompt=generation_prompt_ds, 
        batch_size=1, # Recomendado 1 para evaluar estricto con streaming/análisis
        streaming=False # Deja esto en False para que el log no pese gigabytes de texto plano
    )

except Exception as e:
    print(f"\n[CRITICAL ERROR EN EL PROCESO PRINCIPAL]: {e}")
    import traceback
    print(traceback.format_exc())

finally:
    # 5. Restaurar siempre las salidas originales al terminar (o si truena)
    sys.stdout = original_stdout
    sys.stderr = original_stderr
    print("Logs cerrados y guardados exitosamente.")

Guardando todos los logs de ejecución en: eval_coherence_20260615_230930.log

Iniciando evaluación Fold 1 de 10
Tamaño del set de validación: 17


Evaluating:   0%|          | 0/17 [00:00<?, ?it/s]

Stories ids: tensor([9])


Both `max_new_tokens` (=4096) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/home/isapc/Tesis/unsloth_env/lib/python3.13/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/home/isapc/Tesis/unsloth_env/lib/python3.13/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, 

Y_true: 0
Y_pred: 1
Problematic section found:
2. Problematic sentences:
- 'The new apartment.' !Important:The new apartment.
- 'She said she Would help him Fill Out His Rental Application For Free.'
...

=== USING NEW FORMAT ===
Found 0 sentence lines
Trying alternative pattern...
Alternative pattern found 0 sentences


...

Stories ids: tensor([299])
Y_true: 0
Y_pred: 1
No problematic section found


Evaluating:  65%|######4   | 11/17 [08:19<07:59, 79.85s/it]Both `max_new_tokens` (=4096) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Stories ids: tensor([367])
Y_true: 1
Y_pred: 1
No problematic section found


Evaluating:  71%|#######   | 12/17 [08:20<04:39, 55.83s/it]Both `max_new_tokens` (=4096) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Stories ids: tensor([386])


In [ ]:
# all_folds_results = llm_model.run_cross_validation_evaluation(prompt=generation_prompt_ds, batch_size=1, run_first_fold_only=True, streaming=True)